## The persistent storage abstraction — PV, PVC, StorageClass

For storage that *outlives* the pod, Kubernetes splits the problem in two:

- The **storage admin** provisions storage and exposes it as a **PersistentVolume (PV)** — a *cluster-scoped* object representing a real piece of storage (a 20 GB cloud disk, an NFS share).
- The **app author** asks for storage with a **PersistentVolumeClaim (PVC)** — a *namespaced* object: "I want 5 GiB of `ReadWriteOnce` from the `fast-ssd` class."

A controller binds the two; the pod references the PVC by name.

```
Pod  →references→  PVC ("I want N GiB")  →bound to→  PV ("here's the disk")
                   namespace-scoped                 cluster-scoped
```

The decoupling is the point: the app says *what* it needs, the PV says *what's available*, the binding controller matches them. Apps don't know whether they got EBS, NFS, or Ceph — and don't have to.

### Two ways the PV side is created

- **Static provisioning** — an admin pre-creates PV objects; PVCs bind to the best fit. For hand-curated storage (a corp NFS server, on-prem SAN).
- **Dynamic provisioning** — the admin creates a **StorageClass** describing *how* to make a disk; a PVC references the class and a CSI provisioner **creates a PV on demand.** The everyday cloud case.

Dynamic provisioning needs a StorageClass; most managed clusters ship a **default**. On `kind` the default is `local-path` — a directory on the node, fine for learning, not production. On our map, PV, PVC, and StorageClass are three chips of the **Storage** box, and this triangle is the piece the CKA tests hardest.